In [1]:
import duckdb

In [2]:
# create or connect to a DuckDB database file
conn = duckdb.connect('fleet.db')
print('Connected to fleet.db successfuly!')

Connected to fleet.db successfuly!


### Raw Data Layer

##### Load Raw CSV into a table

In [4]:
# Check schema 
schema = conn.execute("Describe raw_telemetry").fetchdf()
print(schema)

      column_name column_type null   key default extra
0       timestamp   TIMESTAMP  YES  None    None  None
1      vehicle_id     VARCHAR  YES  None    None  None
2       speek_kmh      BIGINT  YES  None    None  None
3  fuel_level_pct      BIGINT  YES  None    None  None
4     distance_km      DOUBLE  YES  None    None  None
5     engine_temp      BIGINT  YES  None    None  None


In [6]:
conn.execute("""CREATE TABLE raw_telemetry AS
                SELECT *
                FROM read_csv_auto('raw_telematry.csv');""")
print("Raw telemetry table created!")

Raw telemetry table created!


#### Inspect data anomalies

In [5]:
# check negative speeds
conn.execute("""SELECT COUNT (*)
                 FROM raw_telemetry
                 WHERE speek_kmh < 0 """).fetchall()

[(0,)]

In [7]:
# check fuel over 100
conn.execute(""" SELECT COUNT (*)
                  FROM raw_telemetry
                  WHERE fuel_level_pct  > 100 """).fetchall()

[(0,)]

In [8]:
# check abnormal engine temps 
conn.execute ("""
SELECT COUNT(*)
FROM raw_telemetry
WHERE engine_temp < 50 OR engine_temp > 140 """).fetchall()

[(0,)]

### Data cleaning Layer

In [11]:
# Create new table (telemetry_clean)
conn.execute("""
CREATE TABLE telemetry_clean AS
SELECT timestamp,
       vehicle_id,
       speek_kmh,
       --- Cap fuel level at 100%
       CASE 
           WHEN fuel_level_pct > 100 THEN 100
           ELSE fuel_level_pct
        END AS fuel_level_pct,
        distance_km,
        engine_temp,
        --- flag temperature anomalies 
        CASE
            WHEN engine_temp < 50 OR engine_temp > 140 THEN 1
            ELSE 0
        END AS temp_anomaly
 FROM  raw_telemetry
 WHERE speek_kmh >= 0  --- remove impossible speeds 
       
       """)

In [12]:
# verify the clean table 
conn.execute("SELECT * FROM telemetry_clean LIMIT 5").fetchdf()

,timestamp,vehicle_id,speek_kmh,fuel_level_pct,distance_km,engine_temp,temp_anomaly
0,2026-01-01 00:00:00,V001,3,34,0.05,91,0
1,2026-01-01 00:10:00,V001,6,42,0.10,110,0
2,2026-01-01 00:20:00,V001,3,39,0.05,93,0
3,2026-01-01 00:30:00,V001,9,91,0.15,104,0
4,2026-01-01 00:40:00,V001,11,31,0.18,100,0


In [13]:
# check total number of records 
conn.execute("SELECT COUNT(*) FROM telemetry_clean").fetchall()

[(150000,)]

In [14]:
# check temp anomalies
conn.execute(""" 
SELECT temp_anomaly, COUNT(*)
FROM telemetry_clean
GROUP BY temp_anomaly
""").fetchall()

[(0, 150000)]

### Dim_Vehicle

In [16]:
# create dimension vehicle table
conn.execute ("""
CREATE TABLE dim_vehicle AS
SELECT DISTINCT vehicle_id
FROM telemetry_clean   
""")

In [17]:
# check total number of vehicles
conn.execute("SELECT COUNT(*) FROM dim_vehicle").fetchall()

[(50,)]

### Dim_Date

In [3]:
conn.execute( """
CREATE TABLE dim_date AS
SELECT DISTINCT
STRFTIME(timestamp, '%Y%m%d'):: INTEGER AS date_key,
CAST(timestamp AS DATE) AS full_date,
EXTRACT(year FROM timestamp) AS Year,
EXTRACT(month FROM timestamp) AS day,
STRFTIME(timestamp, '%w') AS weekday
FROM telemetry_clean;
"""
    
)

In [4]:
# verify dim_date
conn.execute("SELECT * FROM dim_date LIMIT 5").fetchall()

[(20260108, datetime.date(2026, 1, 8), 2026, 1, '4'),
 (20260123, datetime.date(2026, 1, 23), 2026, 1, '5'),
 (20260130, datetime.date(2026, 1, 30), 2026, 1, '5'),
 (20260116, datetime.date(2026, 1, 16), 2026, 1, '5'),
 (20260122, datetime.date(2026, 1, 22), 2026, 1, '4')]

### Fact_Fleet_Metrics

In [21]:
conn.execute("""
CREATE TABLE fact_fleet_metrics AS
WITH fuel_calc AS (
SELECT vehicle_id,
       timestamp,
       distance_km,
       speek_kmh AS speed_kmh ,
       engine_temp,
       fuel_level_pct,
       ---previous fuel reading
       LAG (fuel_level_pct) OVER (PARTITION BY vehicle_id ORDER BY timestamp) AS prev_fuel
FROM telemetry_clean
                   )
SELECT 
      vehicle_id,
      STRFTIME(timestamp, '%Y%m%d'):: INTEGER As date_key,
      ROUND (SUM(distance_km),2)  AS total_distance,
      AVG(speed_kmh) AS avg_speed,
      MAX(speed_kmh) AS max_speed,
      SUM(CASE WHEN speed_kmh > 100 THEN 1 ELSE 0 END) AS overspeed_count,
      SUM(CASE WHEN engine_temp > 110 THEN 1 ELSE 0 END ) AS maintenance_flag,
      --- fuel consumption calculation
      SUM (
          CASE 
            WHEN prev_fuel IS NULL THEN 0
            WHEN prev_fuel > fuel_level_pct THEN prev_fuel - fuel_level_pct
            ELSE 0
          END
          ) AS fuel_consumed_pct
          
FROM fuel_calc
GROUP BY vehicle_id, date_key;
"""
)

In [22]:
# verify fact table
conn.execute("SELECT * FROM fact_fleet_metrics LIMIT 5").fetchall()

[('V042', 20260102, 9.98, 6.02, 12, 0, 0, 1375),
 ('V006', 20260122, 8.91, 5.34, 12, 0, 0, 1239),
 ('V006', 20260125, 9.45, 5.67, 12, 0, 0, 1551),
 ('V028', 20260123, 9.61, 5.76, 12, 0, 0, 1384),
 ('V028', 20260129, 10.78, 6.45, 12, 0, 0, 1427)]

#### Add New Columns to the Fact Table

In [5]:
conn.execute ("""ALTER TABLE fact_fleet_metrics 
                 ADD COLUMN moving_events INTEGER;""")

In [6]:
conn.execute ("""ALTER TABLE fact_fleet_metrics 
                 ADD COLUMN idle_events INTEGER;""")

In [7]:
conn.execute ("""ALTER TABLE fact_fleet_metrics 
                 ADD COLUMN harsh_events INTEGER;""")

#### Update fact table

In [6]:
conn.execute("""UPDATE fact_fleet_metrics f
                SET 
                moving_events = m.moving_events,
                idle_events = m.idle_events,
                harsh_events = m.harsh_driving_events
                FROM (
                SELECT vehicle_id,
                       STRFTIME(timestamp, '%Y%m%d')::INTEGER AS date_key,
                       SUM(CASE WHEN speek_kmh > 0 THEN 1 ELSE 0 END) AS moving_events,
                       SUM(CASE WHEN speek_kmh = 0 AND engine_temp > 70
                           THEN 1 ELSE 0 END
                           ) AS idle_events,
                        SUM (CASE WHEN speek_kmh > 120
                             THEN 1 ELSE 0 END
                             ) AS harsh_driving_events
                FROM telemetry_clean
                GROUP BY vehicle_id, date_key
                ) m
                WHERE f.vehicle_id = m.vehicle_id
                AND f.date_key = m.date_key ;
                """)

In [13]:
# verify
result = conn.execute("""SELECT * FROM fact_fleet_metrics Limit 5 """).fetchdf()
print(result)

  vehicle_id  date_key  total_distance  avg_speed  max_speed  overspeed_count  \
0       V042  20260102            9.98       6.02         12              0.0   
1       V006  20260122            8.91       5.34         12              0.0   
2       V006  20260125            9.45       5.67         12              0.0   
3       V028  20260123            9.61       5.76         12              0.0   
4       V028  20260129           10.78       6.45         12              0.0   

   maintenance_flag  fuel_consumed_pct  moving_events  idle_events  \
0               0.0             1375.0             92            7   
1               0.0             1239.0             89           11   
2               0.0             1551.0             89           11   
3               0.0             1384.0             93            7   
4               0.0             1427.0             96            4   

   harsh_events  
0             0  
1             0  
2             0  
3             0  
4 